<a href="https://colab.research.google.com/github/SSenitha/CCS3052_Advance_DSA/blob/Sandaru-s-Branch/FC211021_aStar_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Dataset

In [ ]:
# Load data from the repository
!wget https://raw.githubusercontent.com/SSenitha/CCS3052_Advance_DSA/refs/heads/main/Cities_of_SriLanka.csv


'wget' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
#Import necessary libraries

import math, time, heapq
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import stats
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

from collections import defaultdict
from sklearn.neighbors import BallTree

In [ ]:
# Read the dataset and output the total count
#df = pd.read_csv('/content/Cities_of_SriLanka.csv')

#----------------For VS Code----------------
df = pd.read_csv('./Cities_of_SriLanka.csv')
#-------------------------------------------

print(f"Total rows count: {df['city id'].count()}")
df.rename(columns={"city id": "city_id"}, inplace=True)
df.head()

Total rows count: 2155


,city_id,district_id,name_en,name_si,name_ta,sub_name_en,sub_name_si,sub_name_ta,postcode,latitude,longitude
0,1,1,Akkaraipattu,අක්කරපත්තුව,அக்கரைப்பற்று,NULL,NULL,NULL,32400.0,7.218428,81.854116
1,2,1,Ambagahawatta,අඹගහවත්ත,அம்பகஹவத்த,NULL,NULL,NULL,90326.0,7.301756,81.674729
2,3,1,Ampara,අම්පාර,அம்பாறை,NULL,NULL,NULL,32000.0,7.301756,81.674729
3,4,1,Bakmitiyawa,බක්මිටියාව,பக்மிடியாவ,NULL,NULL,NULL,32024.0,7.029632,81.680205
4,5,1,Deegawapiya,දීඝවාපිය,தீகவாபி,NULL,NULL,NULL,32006.0,7.301756,81.674729


In [ ]:
# Remove the Cities with little to no geographical difference

df = df.drop_duplicates(subset=['latitude', 'longitude'], keep='first').reset_index(drop=True)
print(f"Total rows count after dropping duplicates: {df['city_id'].count()}")

Total rows count after dropping duplicates: 1919


In [ ]:
# Dropping the columns that won't be used in the making of graph/Matrix
df = df.drop(columns=["district_id","name_si","name_ta","sub_name_en","sub_name_si","sub_name_ta","postcode"])

In [ ]:
def resetIndex():
  df.drop(columns=["city_id"], inplace=True)
  df.insert(0, "city_id", df.index)
  return df

In [ ]:
resetIndex()
df.head()

,city_id,name_en,latitude,longitude
0,0,Akkaraipattu,7.218428,81.854116
1,1,Ambagahawatta,7.301756,81.674729
2,2,Bakmitiyawa,7.029632,81.680205
3,3,Digamadulla Weeragoda,7.390125,81.696588
4,4,Dorakumbura,7.358870,81.301428


# Adjacency/Sparse Matrix Approach

In [ ]:
k = 6
num_cities = len(df)

# Convert lat/lon to radians for haversine
X_rad = np.radians(df[['latitude', 'longitude']].values)
nn = NearestNeighbors(n_neighbors=k, algorithm='ball_tree', metric='haversine').fit(X_rad)

# Find the 5 nearest neighbors
distances_rad, indices = nn.kneighbors(X_rad)

# Convert to km
R = 6371.0088
distances = distances_rad * R

rows, cols, data = [], [], []
seen_edges = set()

for i in range(num_cities):
    for j in range(1, k):
        n_index = indices[i, j]

        edge = (min(i, n_index), max(i, n_index))
        if edge in seen_edges:
            continue
        seen_edges.add(edge)

        n_distance = distances[i, j]

        rows.extend([i, n_index])
        cols.extend([n_index, i])
        data.extend([n_distance, n_distance])

sparse_matrix = csr_matrix((data, (rows, cols)), shape=(num_cities, num_cities))

print("Shape of the sparse adjacency matrix:", sparse_matrix.shape)

Shape of the sparse adjacency matrix: (1919, 1919)


In [ ]:
print("\nFirst 10 rows and their non-zero entries (neighbors and distances):")

for i in range(min(10, num_cities)):
    row_data = sparse_matrix.getrow(i)
    print(f"Row {i}: {list(zip(row_data.indices, row_data.data))}")


First 10 rows and their non-zero entries (neighbors and distances):
Row 0: [(np.int32(19), np.float64(11.431624263526398)), (np.int32(20), np.float64(8.483954138345275)), (np.int32(32), np.float64(10.016577346523386)), (np.int32(33), np.float64(11.467365056871419)), (np.int32(237), np.float64(2.8731802094217413)), (np.int32(256), np.float64(2.1788768806124006))]
Row 1: [(np.int32(3), np.float64(10.11756911009424)), (np.int32(15), np.float64(10.123962927467872)), (np.int32(24), np.float64(8.746601354211169)), (np.int32(34), np.float64(8.084691843487471)), (np.int32(857), np.float64(8.321695251073267))]
Row 2: [(np.int32(6), np.float64(11.596172893263102)), (np.int32(13), np.float64(20.933421754060213)), (np.int32(14), np.float64(18.1332450176922)), (np.int32(35), np.float64(15.95628286236248)), (np.int32(736), np.float64(17.196762745885444)), (np.int32(1287), np.float64(16.261707350785333))]
Row 3: [(np.int32(1), np.float64(10.11756911009424)), (np.int32(5), np.float64(15.8255307407874

# A* Algorithm for shortest path

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0088  # Earth's radius in km
    phi1 = math.radians(lat1); phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1); dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2.0)**2
    a = min(1.0, max(0.0, a))  # safety
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
def dist_index(index1, index2):
    lat1 = df.loc[index1, 'latitude']
    lon1 = df.loc[index1, 'longitude']
    lat2 = df.loc[index2, 'latitude']
    lon2 = df.loc[index2, 'longitude']

    return haversine(lat1, lon1, lat2, lon2)

In [ ]:
def get_neighbors(city_index):
    row_data = sparse_matrix.getrow(city_index)
    return list(row_data.indices)

In [ ]:
def get_neighbors_with_distances(index, sparse_matrix):
    # Efficiently get neighbors and distances from CSR matrix
    row = sparse_matrix[index]
    neighbors = row.indices
    distances = row.data
    return zip(neighbors, distances)

def reconstruct_path(came_from, current):
    path = [current]
    while current in came_from:
        current = came_from[current]
        path.append(current)
    return path[::-1]

def astar(start_index, end_index, sparse_matrix):
    open_set = []
    heapq.heappush(open_set, (0, start_index))

    g_costs = {start_index: 0}
    came_from = {}

    while open_set:
        f_cost, current_index = heapq.heappop(open_set)

        if current_index == end_index:
            path = reconstruct_path(came_from, current_index)
            return path, g_costs[end_index]

        for neighbor_index, edge_weight in get_neighbors_with_distances(current_index, sparse_matrix):
            tentative_g = g_costs[current_index] + edge_weight

            if neighbor_index not in g_costs or tentative_g < g_costs[neighbor_index]:
                g_costs[neighbor_index] = tentative_g
                came_from[neighbor_index] = current_index

                # Use your existing haversine-based dist_index() here
                heuristic = dist_index(neighbor_index, end_index)
                f = tentative_g + heuristic
                heapq.heappush(open_set, (f, neighbor_index))

    return None, None  # No path found


In [ ]:
import sys

# Get start city
search_name = input("Enter start city: ").strip().lower()
start = df[df['name_en'].str.lower() == search_name]

if start.empty:
    print("No city found with that name.")
    sys.exit(0)

# Get end city
search_name = input("Enter end city: ").strip().lower()
end = df[df['name_en'].str.lower() == search_name]

if end.empty:
    print("No city found with that name.")
    sys.exit(0)

# Get city IDs
start_id = start['city_id'].values[0]
end_id = end['city_id'].values[0]

# Run A*
lat_radians = np.radians(df['latitude'].values)
lon_radians = np.radians(df['longitude'].values)

start_time = time.perf_counter()
shortest_path, total_distance = astar(start_id, end_id, sparse_matrix)
end_time = time.perf_counter()
execution_time = end_time - start_time

# Print the shortest path
if shortest_path:
    print("Shortest path found:")
    city_names = [df.loc[i, 'name_en'] for i in shortest_path]
    print(" -> ".join(city_names))

    print(f"Cities visited: {len(city_names)}")

    print(f"Total distance: {total_distance:.2f} km")

    print(f"Execution time: {execution_time:.4f} seconds")
else:
    print(f"No path found between city index {start_id} and {end_id}.")

Shortest path found:
Mullativu -> Mulliyawalai -> Oddusudan -> Nainamadu -> Puliyankulam -> Omanthai -> Thandikulam -> Vavuniya
Cities visited: 8
Total distance: 71.75 km
Execution time: 0.0017 seconds
